## LangFuse 실습

In [1]:
import os
import pathlib
import sys

here = pathlib.Path.cwd()
ROOT = here.parents[2] if here.name == "day01" else here
os.chdir(ROOT)
SANDBOX = ROOT / "sandbox" / "w4" / "day01"

print("프로젝트 루트  :", ROOT)

if str(ROOT / 'backend') not in sys.path:
    sys.path.insert(0, str(ROOT / 'backend'))

프로젝트 루트  : /Users/pyoyoung-gyu/Desktop/Personal Project/한화아카데미/AI 서비스 백엔드 프로그래밍 실무/hanwha-agent/agent_practice


* v2방식 (실행 -10분 이후 확인)

In [2]:
import sys
from datetime import datetime, timezone
from pathlib import Path

import anthropic  # noqa: E402
from langfuse import Langfuse  # noqa: E402, v2버전 (로컬용))

from app.core.config import get_settings  # noqa: E402

# 연습 질문과 프롬프트
QUESTION = "부산 출장 숙박비 한도가 얼마인가요?"
SYSTEM = "너는 사내 규정 질의응답 도우미다. 근거가 없으면 없다고 말한다."

# 설정 정보 가져오기
settings = get_settings()

# 키 존재 여부 확인
if not settings.langfuse_public_key or settings.langfuse_secret_key is None:
    print(".env 의 LANGFUSE_PUBLIC_KEY / LANGFUSE_SECRET_KEY 가 비어 있습니다.")
    print("키를 붙여 넣고 저장했는지, 레포 루트에서 실행했는지 확인하세요.")
    sys.exit(1)

# 랭퓨즈 객체 생성
lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key.get_secret_value(),
    host=settings.langfuse_host,
)
print("보낼 곳 :", settings.langfuse_host)

# 키가 해당 주소에서 통하는 키인지 확인
try:
    lf.auth_check()
except Exception as e: 
    print("인증 실패 :", type(e).__name__)
    print("  ① 키를 발급한 리전과 LANGFUSE_HOST 가 같은가 (jp.cloud · cloud · us.cloud)")
    print("  ② pk-lf- 와 sk-lf- 를 서로 바꿔 붙이지 않았는가")
    print("  ③ 키 앞뒤에 공백이나 따옴표가 붙지 않았는가")
    print("  ④ localhost 라면 컨테이너가 떠 있는가 (docker compose ps)")
    sys.exit(1)
print("인증     : 통과")

# 트레이스 하나와 그 안에 제네레이션 하나 생성
trace = lf.trace(
    name="trace-hello",
    input=QUESTION,
    tags=["w04d01", "hello"],
    metadata={"host": settings.langfuse_host},
)
generation = trace.generation(
    name="claude",
    model=settings.llm_model,
    input=[{"role": "system", "content": SYSTEM},
           {"role": "user", "content": QUESTION}],
    start_time=datetime.now(timezone.utc),
)

# 클로드 호출
key = settings.anthropic_api_key
try:
    if key is None:
        raise RuntimeError(".env 의 ANTHROPIC_API_KEY 가 비어 있습니다")
    # 클라이언트 생성
    client = anthropic.Anthropic(api_key=key.get_secret_value())
    resp = client.messages.create( 
        model=settings.llm_model,
        max_tokens=settings.max_tokens,
        system=SYSTEM,
        messages=[{"role": "user", "content": QUESTION}],
    )
    answer = "".join(b.text for b in resp.content if b.type == "text")
    # 응답 시 제너레이션에 값 추가
    generation.end(
        output=answer,
        usage_details={"input": resp.usage.input_tokens,
                       "output": resp.usage.output_tokens},
    )
    # 트레이스에 값 업데이트
    trace.update(output=answer)
    print("Claude   :", answer[:60].replace("\n", " "), "...")
    print("토큰     : 입력", resp.usage.input_tokens, "· 출력", resp.usage.output_tokens)
except Exception as e:  
    generation.end(level="ERROR", status_message=f"{type(e).__name__}: {e}")
    trace.update(output=None)
    print("Claude   : 호출 실패 -", type(e).__name__)
    print("           그래도 트레이스는 보냅니다. 화면에서 빨간 ERROR 로 보입니다.")

# 랭퓨즈 밀어넣기(실제로 랭퓨즈에 보내기)
lf.flush()

# 화면 주소 생성 (픤의성, 옵션)
base = settings.langfuse_host.rstrip("/")
try:
    project_id = lf.client.projects.get().data[0].id  
    print("트레이스 :", f"{base}/project/{project_id}/traces/{trace.id}")
except Exception: 
    print("트레이스 :", trace.id, "(화면 왼쪽 Tracing → Traces 에서 찾으세요)")


보낼 곳 : http://localhost:3000
인증     : 통과
Claude   : 죄송하지만, 현재 제가 접근할 수 있는 사내 규정 자료에 부산 출장 숙박비 한도에 대한 정보가 없습니다.   ...
토큰     : 입력 66 · 출력 146
트레이스 : http://localhost:3000/project/cmuat43ps0006a7shbis9lyqj/traces/74286928-1911-4fe0-812d-cbfddf214c7d


* v4 계열 문법(cloud) - 지금 당장 실행 x

In [ ]:
lf = Langfuse(
    public_key=settings.langfuse_public_key,
    secret_key=settings.langfuse_secret_key.get_secret_value(),
    host=settings.langfuse_host,
)

# v4 계열 문법(Clause)
with lf.start_as_current_observation(
    as_type="span",
    name="trace-hello",
    input=QUESTION
)as trace:
    with lf.start_as_current_obervation(
        as_type="generation",
        name="claude",
        model=settings.llm_model,
        input=[
            {"role": "system", "content": SYSTEM},
            {"role": "user", "content": QUESTION},
        ]
    ) as generation:
        # Claude호출
        generation.update(output=answer)
    trace.update(output=answer)

lf.flush()

SyntaxError: ':' expected after dictionary key (2470010076.py, line 18)

In [8]:
from app.integrations import langfuse_client

print("get_client() : ", langfuse_client.get_client())

with langfuse_client.trace("ask", run_id="RUN-1234") as handle:
    print("trace()의 handle : ", handle)
    print(f"with 블록 안 계산 : 3 + 4 = {3+4}")

langfuse_client.score("RUN-1234", "golden_pass", 1.0)
print("score()호출 : 예외 없음")

get_client() :  None
trace()의 handle :  None
with 블록 안 계산 : 3 + 4 = 7
score()호출 : 예외 없음


In [3]:
import app.integrations.factory as factory
from sqlalchemy import select

from app.db.session import session_scope
from app.integrations.ports import LLMResult
from app.models import Run
from app.services import chat_service

# fake 답변
REPLY = ('{"answer": "부산 출장 숙박비는 1박 7만원 이내입니다.", '
         '"sources": [{"doc_id": "DOC-HR-014", "title": "국내출장 여비 규정", '
         '"version": "v2.0", "locator": "제12조(숙박비) · p.6"}], '
         '"enough_evidence": true}')


class StubLLM:

    name = "stub"

    def answer(self, *, question: str, contexts: list[dict], user: dict) -> LLMResult:
        return LLMResult(text=REPLY, model="claude-haiku-4-5",
                         input_tok=1200, output_tok=300, cost_krw=3.8, latency_ms=900)


original = factory.get_llm
try:
    factory.get_llm = lambda: StubLLM()
    out = chat_service.ask(question="부산 출장 숙박비 한도가 얼마인가요?")
finally:
    factory.get_llm = original

print("ask() 가 돌려준 run_id :", out.run_id)
print()

with session_scope() as session:
    run = session.scalars(select(Run).where(Run.id == out.run_id)).one()
    print("runs 표에서 찾은 행")
    print("  id         :", run.id)
    print("  user_id    :", run.user_id, "       (김민준 · 인프라사업부 2팀 · 사번 2019-0412)")
    print("  status     :", run.status)
    print("  latency_ms :", run.latency_ms)
    print("  answer     :", run.answer[:16] + "...")

ask() 가 돌려준 run_id : RUN-8823

runs 표에서 찾은 행
  id         : RUN-8823
  user_id    : 1        (김민준 · 인프라사업부 2팀 · 사번 2019-0412)
  status     : 완료
  latency_ms : 7
  answer     : 부산 출장 숙박비는 1박 7만...


In [7]:
from fastapi.testclient import TestClient

from app.main import app

client = TestClient(app)

r_ok = client.get("/api/v1/chat/runs/RUN-8821")
print("status code : ", r_ok.status_code)
print("question : ", r_ok.json()["question"])
print("status : ", r_ok.json()["status"])

r_no = client.get("/api/v1/chat/runs/RUN-9999")
print("RUN-9999 code : ", r_no.status_code)

17:32:24 INFO     httpx2 : HTTP Request: GET http://testserver/api/v1/chat/runs/RUN-8821 "HTTP/1.1 200 OK"
status code :  200
question :  부산 출장 숙박비 한도가 얼마인가요?
status :  완료
17:32:24 INFO     httpx2 : HTTP Request: GET http://testserver/api/v1/chat/runs/RUN-9999 "HTTP/1.1 404 Not Found"
RUN-9999 code :  404
